# Notebook 2 — Data Cleaning\n\n**Assam Government Procurement Analytics**\n\nThis notebook loads the raw `tenders` table from `db/assam_procurement.db` (created in Notebook 1) and cleans it into an analysis-ready table.\n\n**What this notebook does:**\n1. Loads `tenders` from SQLite.\n2. Drops columns that carry no information (100% null).\n3. Fixes `tender_value_amount`, which is stored as Indian-comma-formatted text (e.g. `1,59,98,681`) instead of plain numbers.\n4. Standardises the two date columns to real `datetime` values.\n5. Converts remaining numeric-but-text columns to actual numbers.\n6. Verifies the existing `fiscal_year` column against the dates.\n7. Adds a handful of small derived columns useful for analysis.\n8. Saves the result as a new table, **`tenders_clean`**, in SQLite, and as `data/processed/tenders_clean.csv`.\n9. Prints a full before/after cleaning summary.\n\nNo rows are removed in this notebook — only columns are dropped, retyped, or added. Row-level filtering (if any) belongs in the analysis notebook, not here.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
DB_PATH = PROJECT_ROOT / "db" / "assam_procurement.db"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Database:", DB_PATH, "| exists:", DB_PATH.exists())
print("Processed output dir:", PROCESSED_DIR)

Database: C:\Users\Avni\assam-procurement-analytics\db\assam_procurement.db | exists: True
Processed output dir: C:\Users\Avni\assam-procurement-analytics\data\processed


## Step 1 — Load the raw `tenders` table\n\nWe pull the entire table out of SQLite with a plain SQL `SELECT *` and load it into a DataFrame to work on. We keep a copy of the starting shape (`rows_before`, `cols_before`) so the summary at the end of this notebook can report exactly what changed.

In [2]:
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("SELECT * FROM tenders;", conn)

rows_before, cols_before = df.shape
columns_before = list(df.columns)

print(f"Loaded 'tenders': {rows_before} rows, {cols_before} columns")
df.dtypes

Loaded 'tenders': 34232 rows, 26 columns


_link                                  object
id                                     object
tag                                    object
date                                   object
ocid                                   object
Payment Mode                           object
initiationType                         object
fiscal_year                            object
buyer_name                             object
tender_id                              object
tender_stage                           object
tender_title                           object
tender_allowTwoStageTender             object
tender_status                          object
tender_submissionMethodDetails         object
tender_procurementMethod               object
tender_mainProcurementCategory         object
tender_contractType                    object
tender_numberOfTenderers              float64
tender_datePublished                   object
tender_allowPreferentialBidder         object
tender_externalReference          

## Step 2 — Drop fully-null columns\n\n`tender_status` and `tender_submissionMethodDetails` are null for every single row. A column that is 100% missing carries zero information and only adds noise to later analysis, so we drop both. We first assert they really are fully null (as a safety check against silently dropping a column that turns out to have data), then drop them and record the drop for the cleaning summary.

In [3]:
fully_null_columns = ["tender_status", "tender_submissionMethodDetails"]

for col in fully_null_columns:
    null_count = df[col].isnull().sum()
    assert null_count == len(df), (
        f"Expected '{col}' to be 100% null, but only {null_count}/{len(df)} rows are null. "
        "Investigate before dropping."
    )
    print(f"Confirmed '{col}' is null in all {null_count} rows.")

df = df.drop(columns=fully_null_columns)
columns_dropped = list(fully_null_columns)

print("\nColumns dropped:", columns_dropped)
print("Shape after drop:", df.shape)

Confirmed 'tender_status' is null in all 34232 rows.
Confirmed 'tender_submissionMethodDetails' is null in all 34232 rows.

Columns dropped: ['tender_status', 'tender_submissionMethodDetails']
Shape after drop: (34232, 24)


## Step 3 — Clean `tender_value_amount`\n\nThis column looks numeric but is stored as **text using Indian comma grouping**, e.g. `\"1,59,98,681\"` instead of `15998681`. If we convert it to numeric naively (`pd.to_numeric` directly on the raw text), pandas can't parse the commas and silently turns every comma-formatted value into `NaN` — inflating the apparent null count from a true **1,580** blanks to a misleading **5,617** \"nulls\".\n\nThe fix: strip the commas out of the text *before* converting to numeric. This recovers the ~4,037 values that were never actually missing — they were just formatted with thousand separators. Whatever is left null after that conversion is a *genuine* missing value, which we keep as `NaN` (no imputation) and mark with a new boolean flag column, `tender_value_amount_missing`, so downstream analysis can see at a glance which rows lack a value.

In [4]:
naive_nulls = pd.to_numeric(df["tender_value_amount"], errors="coerce").isnull().sum()

# Strip thousand-separator commas from the text, but only where the value
# isn't already null (str.replace on NaN would turn it into the string "nan").
value_no_commas = df["tender_value_amount"].where(
    df["tender_value_amount"].isnull(),
    df["tender_value_amount"].astype(str).str.replace(",", "", regex=False),
)
df["tender_value_amount"] = pd.to_numeric(value_no_commas, errors="coerce")

true_nulls = df["tender_value_amount"].isnull().sum()
df["tender_value_amount_missing"] = df["tender_value_amount"].isnull()

print(f"Nulls from naive to_numeric (commas break parsing): {naive_nulls}")
print(f"Nulls after stripping commas first (genuine missing values): {true_nulls}")
print(f"Values recovered by fixing the comma formatting:           {naive_nulls - true_nulls}")

Nulls from naive to_numeric (commas break parsing): 5617
Nulls after stripping commas first (genuine missing values): 1580
Values recovered by fixing the comma formatting:           4037


## Step 4 — Standardise date columns\n\nTwo columns hold dates as text, in different formats:\n\n- **`tender_bidOpening_date`** — day-first format, `DD-MM-YYYY H:MM` (e.g. `20-11-2017 11:00`), with a small number of rows using a single-digit hour (`9:00` instead of `09:00`). We parse with `dayfirst=True` so `20-11-2017` is read as 20 November, not the 20th month.\n- **`tender_datePublished`** — already in ISO-like order, `YYYY-MM-DD HH:MM:SS`.\n\nBoth are converted with `errors=\"coerce\"` so that any value that still fails to parse becomes `NaT` (a proper missing-date marker) rather than crashing the notebook. We then check how many rows failed to parse in each column — that count should be 0 for both.

In [5]:
df["tender_bidOpening_date"] = pd.to_datetime(
    df["tender_bidOpening_date"], dayfirst=True, errors="coerce"
)
df["tender_datePublished"] = pd.to_datetime(
    df["tender_datePublished"], errors="coerce"
)

print("tender_bidOpening_date parse failures:", df["tender_bidOpening_date"].isnull().sum())
print("tender_datePublished parse failures:  ", df["tender_datePublished"].isnull().sum())
print()
print("Sample parsed dates:")
df[["tender_bidOpening_date", "tender_datePublished"]].head()

tender_bidOpening_date parse failures: 0
tender_datePublished parse failures:   0

Sample parsed dates:


,tender_bidOpening_date,tender_datePublished
0,2016-07-12 12:30:00,2016-06-16 12:00:00
1,2017-01-02 14:00:00,2016-12-12 18:00:00
2,2017-01-04 14:05:00,2016-12-15 09:00:00
3,2017-01-11 14:05:00,2016-12-28 18:00:00
4,2017-01-11 14:00:00,2016-12-22 18:00:00


## Step 5 — Convert remaining numeric columns\n\n`tender_numberOfTenderers` and `tender_tenderPeriod_durationInDays` should be numbers but arrive from SQLite without a guaranteed dtype, so we explicitly coerce both with `pd.to_numeric`. `tender_numberOfTenderers` has genuine gaps (a tender with no recorded bidder count yet) — those stay as `NaN`, consistent with how we handled `tender_value_amount`.

In [6]:
df["tender_numberOfTenderers"] = pd.to_numeric(df["tender_numberOfTenderers"], errors="coerce")
df["tender_tenderPeriod_durationInDays"] = pd.to_numeric(
    df["tender_tenderPeriod_durationInDays"], errors="coerce"
)

print("tender_numberOfTenderers nulls:          ", df["tender_numberOfTenderers"].isnull().sum())
print("tender_tenderPeriod_durationInDays nulls:", df["tender_tenderPeriod_durationInDays"].isnull().sum())

tender_numberOfTenderers nulls:           2573
tender_tenderPeriod_durationInDays nulls: 0


## Step 6 — Verify `fiscal_year` against `tender_datePublished`\n\n`fiscal_year` is already present in the data (e.g. `\"2017-2018\"`). Rather than trust it blindly, we independently derive a fiscal year from `tender_datePublished` using the Indian government fiscal year convention — **1 April to 31 March** — and compare it to the existing column. We don't overwrite `fiscal_year`; this is purely a validation check, and we report how many rows (if any) disagree.

In [7]:
def derive_fiscal_year(published_date):
    """Indian government fiscal year: 1 April to 31 March."""
    if pd.isnull(published_date):
        return None
    year = published_date.year
    if published_date.month >= 4:
        return f"{year}-{year + 1}"
    return f"{year - 1}-{year}"


derived_fiscal_year = df["tender_datePublished"].apply(derive_fiscal_year)
mismatches = derived_fiscal_year.notnull() & (derived_fiscal_year != df["fiscal_year"])

print(f"Rows checked:     {df['tender_datePublished'].notnull().sum()}")
print(f"fiscal_year mismatches: {mismatches.sum()}")

if mismatches.sum() > 0:
    print("\nSample mismatches:")
    display(
        df.loc[mismatches, ["tender_id", "tender_datePublished", "fiscal_year"]]
        .assign(derived_fiscal_year=derived_fiscal_year[mismatches])
        .head(10)
    )
else:
    print("fiscal_year matches the date-derived value for every row with a valid publish date.")

Rows checked:     34232
fiscal_year mismatches: 0
fiscal_year matches the date-derived value for every row with a valid publish date.


## Step 7 — Derived columns\n\nThree small columns that make later analysis simpler:\n\n- **`value_crores`** — `tender_value_amount` divided by 1,00,00,000 (1 crore = 10,000,000), since Indian government tender values are conventionally discussed in crores.\n- **`is_low_competition`** — `True` when exactly one tenderer bid (a classic red flag for weak competition).\n- **`is_nonopen`** — `True` when the procurement method is anything other than `\"Open Tender\"` (i.e. Limited, Single-source, etc. — less competitive procurement routes).

In [8]:
df["value_crores"] = df["tender_value_amount"] / 10_000_000
df["is_low_competition"] = df["tender_numberOfTenderers"] == 1
df["is_nonopen"] = df["tender_procurementMethod"] != "Open Tender"

print("is_low_competition True count:", df["is_low_competition"].sum())
print("is_nonopen True count:        ", df["is_nonopen"].sum())
df[["tender_value_amount", "value_crores", "tender_numberOfTenderers", "is_low_competition",
    "tender_procurementMethod", "is_nonopen"]].head()

is_low_competition True count: 4532
is_nonopen True count:         468


,tender_value_amount,value_crores,tender_numberOfTenderers,is_low_competition,tender_procurementMethod,is_nonopen
0,25132914.0,2.513291,5.0,False,Open Tender,False
1,25349923.0,2.534992,7.0,False,Open Tender,False
2,7967277.0,0.796728,5.0,False,Open Tender,False
3,9803783.0,0.980378,10.0,False,Open Tender,False
4,12441168.0,1.244117,10.0,False,Open Tender,False


## Step 8 — Save the cleaned data\n\nThe cleaned DataFrame is written to two places:\n\n1. A new SQLite table, **`tenders_clean`**, in the same database (via `to_sql`, again with `if_exists=\"replace\"` for safe re-runs). The original `tenders` table is left untouched as the raw record.\n2. A CSV export at `data/processed/tenders_clean.csv`, for anyone who wants the cleaned data without touching SQLite.\n\nOne SQLite-specific note: SQLite has no native `datetime` or `boolean` type. `to_sql` stores our `datetime64` columns as ISO-format text and our boolean columns as `0`/`1` integers. This is standard and lossless — Notebook 3 will parse the date columns back into `datetime` after reading them from SQL.

In [9]:
df.to_sql("tenders_clean", conn, if_exists="replace", index=False)
conn.commit()

csv_path = PROCESSED_DIR / "tenders_clean.csv"
df.to_csv(csv_path, index=False)

rows_after, cols_after = df.shape
columns_after = list(df.columns)

print(f"Saved 'tenders_clean' table to {DB_PATH} ({rows_after} rows, {cols_after} columns)")
print(f"Saved CSV to {csv_path}")

Saved 'tenders_clean' table to C:\Users\Avni\assam-procurement-analytics\db\assam_procurement.db (34232 rows, 28 columns)
Saved CSV to C:\Users\Avni\assam-procurement-analytics\data\processed\tenders_clean.csv


## Step 9 — Verify the save\n\nA quick round-trip check with real SQL: query `tenders_clean` back out of the database and confirm the row count matches what we just wrote, and that new columns like `value_crores` are present.

In [10]:
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM tenders_clean;")
db_row_count = cursor.fetchone()[0]

sample_from_db = pd.read_sql_query(
    "SELECT tender_id, tender_value_amount, value_crores, is_low_competition, is_nonopen "
    "FROM tenders_clean LIMIT 5;",
    conn,
)

assert db_row_count == rows_after, "Row count in SQLite doesn't match the DataFrame just written."
print(f"'tenders_clean' round-trip row count: {db_row_count} (matches DataFrame: {rows_after})")
print()
display(sample_from_db)

conn.close()

'tenders_clean' round-trip row count: 34232 (matches DataFrame: 34232)



,tender_id,tender_value_amount,value_crores,is_low_competition,is_nonopen
0,2016_DOT_946_1,25132914.0,2.513291,0,0
1,2016_DoWR_1302_1,25349923.0,2.534992,0,0
2,2016_DoWR_1318_1,7967277.0,0.796728,0,0
3,2016_DoWR_1334_1,9803783.0,0.980378,0,0
4,2016_DoWR_1366_1,12441168.0,1.244117,0,0


## Cleaning Summary\n\nA consolidated before/after report: row and column counts, exactly which columns were dropped or added, and how many nulls remain in every column of the final `tenders_clean` table.

In [11]:
columns_added = [c for c in columns_after if c not in columns_before]

print("=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)
print(f"Rows before:    {rows_before}")
print(f"Rows after:     {rows_after}  (no rows were removed — cleaning only touched columns)")
print()
print(f"Columns before: {cols_before}")
print(f"Columns after:  {cols_after}")
print()
print(f"Columns dropped ({len(columns_dropped)}): {columns_dropped}")
print(f"Columns added ({len(columns_added)}):   {columns_added}")
print()
print("Nulls remaining per column in 'tenders_clean':")
print("-" * 60)

null_counts = df.isnull().sum().sort_values(ascending=False)
display(null_counts.to_frame(name="null_count"))

CLEANING SUMMARY
Rows before:    34232
Rows after:     34232  (no rows were removed — cleaning only touched columns)

Columns before: 26
Columns after:  28

Columns dropped (2): ['tender_status', 'tender_submissionMethodDetails']
Columns added (4):   ['tender_value_amount_missing', 'value_crores', 'is_low_competition', 'is_nonopen']

Nulls remaining per column in 'tenders_clean':
------------------------------------------------------------


,null_count
tender_numberOfTenderers,2573
tender_value_amount,1580
value_crores,1580
tender_stage,386
date,0
ocid,0
initiationType,0
Payment Mode,0
fiscal_year,0
_link,0
